# Лабораторная работа №2. Логистическая и линейная регрессия

Исследование алгоритма линейной и логистической регрессии на задачах классификации и регрессии.

Датасеты те же, что и в первой работе: Wine для классификации и Parkinsons Telemonitoring для регрессии.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import mean_squared_error, r2_score, ConfusionMatrixDisplay
from pathlib import Path
import sys
for candidate in [Path.cwd() / "src", Path.cwd().parent / "src"]:
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass
%matplotlib inline

## Загрузка данных

In [ ]:
# Wine
wine_cols = ['class', 'alcohol', 'malic_acid', 'ash', 'alcalinity', 'magnesium',
             'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins',
             'color_intensity', 'hue', 'od280', 'proline']
wine = pd.read_csv('../data/wine.data', header=None, names=wine_cols)

# Parkinsons
parkinsons = pd.read_csv('../data/parkinsons_updrs.data')

X_wine = wine.drop('class', axis=1)
y_wine = wine['class']
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine)

X_park = parkinsons.drop(['subject#', 'motor_UPDRS', 'total_UPDRS'], axis=1)
y_park = parkinsons['total_UPDRS']
X_park_train, X_park_test, y_park_train, y_park_test = train_test_split(
    X_park, y_park, test_size=0.2, random_state=42)

print('Данные загружены')
print('Wine:', X_wine_train.shape, X_wine_test.shape)
print('Parkinsons:', X_park_train.shape, X_park_test.shape)


## 2. Создание бейзлайна и оценка качества

Обучаю модели из sklearn с параметрами по умолчанию.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

# Классификация
clf_base = LogisticRegression()
clf_base.fit(X_wine_train, y_wine_train)
y_wine_pred_base = clf_base.predict(X_wine_test)

print('Классификация (бейзлайн)')
print('Accuracy:', round(accuracy_score(y_wine_test, y_wine_pred_base), 4))
print('F1-macro:', round(f1_score(y_wine_test, y_wine_pred_base, average='macro'), 4))

# Регрессия
reg_base = LinearRegression()
reg_base.fit(X_park_train, y_park_train)
y_park_pred_base = reg_base.predict(X_park_test)

print('\nРегрессия (бейзлайн)')
print('RMSE:', round(np.sqrt(mean_squared_error(y_park_test, y_park_pred_base)), 4))
print('R2:', round(r2_score(y_park_test, y_park_pred_base), 4))


## 3. Улучшение бейзлайна

Гипотезы:
- Стандартизация признаков должна помочь
- Подбор гиперпараметров на кросс-валидации улучшит результат


In [ ]:
# Стандартизация
scaler_wine = StandardScaler()
X_wine_train_scaled = scaler_wine.fit_transform(X_wine_train)
X_wine_test_scaled = scaler_wine.transform(X_wine_test)

scaler_park = StandardScaler()
X_park_train_scaled = scaler_park.fit_transform(X_park_train)
X_park_test_scaled = scaler_park.transform(X_park_test)

# Подбор параметров для логистической и линейной регрессии
param_grid_clf = {'C': [0.01, 0.1, 1, 10], 'max_iter': [1000]}
param_grid_reg = {'fit_intercept': [True, False]}

grid_clf = GridSearchCV(LogisticRegression(), param_grid_clf, cv=5, scoring='f1_macro')
grid_clf.fit(X_wine_train_scaled, y_wine_train)
print('Лучшие параметры классификации:', grid_clf.best_params_)

grid_reg = GridSearchCV(LinearRegression(), param_grid_reg, cv=5, scoring='neg_root_mean_squared_error')
grid_reg.fit(X_park_train_scaled, y_park_train)
print('Лучшие параметры регрессии:', grid_reg.best_params_)

# Оценка улучшенных моделей
clf_imp = grid_clf.best_estimator_
y_wine_pred_imp = clf_imp.predict(X_wine_test_scaled)

print('\nКлассификация (улучшенный)')
print('Accuracy:', round(accuracy_score(y_wine_test, y_wine_pred_imp), 4))
print('F1-macro:', round(f1_score(y_wine_test, y_wine_pred_imp, average='macro'), 4))

reg_imp = grid_reg.best_estimator_
y_park_pred_imp = reg_imp.predict(X_park_test_scaled)

print('\nРегрессия (улучшенный)')
print('RMSE:', round(np.sqrt(mean_squared_error(y_park_test, y_park_pred_imp)), 4))
print('R2:', round(r2_score(y_park_test, y_park_pred_imp), 4))


## 4. Собственная реализация

Использую свою реализацию алгоритма.


In [ ]:
from linear_logistic import LogisticRegression as OwnLogisticRegression, LinearRegression as OwnLinearRegression

# Классификация
clf_own = OwnLogisticRegression()
clf_own.fit(X_wine_train_scaled, y_wine_train)
y_wine_pred_own = clf_own.predict(X_wine_test_scaled)

print('Собственная реализация (классификация)')
print('Accuracy:', round(accuracy_score(y_wine_test, y_wine_pred_own), 4))
print('F1-macro:', round(f1_score(y_wine_test, y_wine_pred_own, average='macro'), 4))

# Регрессия
reg_own = OwnLinearRegression()
reg_own.fit(X_park_train_scaled, y_park_train)
y_park_pred_own = reg_own.predict(X_park_test_scaled)

print('\nСобственная реализация (регрессия)')
print('RMSE:', round(np.sqrt(mean_squared_error(y_park_test, y_park_pred_own)), 4))
print('R2:', round(r2_score(y_park_test, y_park_pred_own), 4))

## Сравнение результатов

In [ ]:
results = pd.DataFrame({
    'Модель': ['Sklearn Baseline', 'Sklearn Improved', 'Own Implementation'],
    'Accuracy': [
        accuracy_score(y_wine_test, y_wine_pred_base),
        accuracy_score(y_wine_test, y_wine_pred_imp),
        accuracy_score(y_wine_test, y_wine_pred_own)
    ],
    'F1-macro': [
        f1_score(y_wine_test, y_wine_pred_base, average='macro'),
        f1_score(y_wine_test, y_wine_pred_imp, average='macro'),
        f1_score(y_wine_test, y_wine_pred_own, average='macro')
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_park_test, y_park_pred_base)),
        np.sqrt(mean_squared_error(y_park_test, y_park_pred_imp)),
        np.sqrt(mean_squared_error(y_park_test, y_park_pred_own))
    ],
    'R2': [
        r2_score(y_park_test, y_park_pred_base),
        r2_score(y_park_test, y_park_pred_imp),
        r2_score(y_park_test, y_park_pred_own)
    ]
})
print(results.round(4))

# Графики
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
results.set_index('Модель')[['Accuracy', 'F1-macro']].plot(kind='bar', ax=axes[0], rot=15)
axes[0].set_title('Классификация')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')

results.set_index('Модель')['RMSE'].plot(kind='bar', ax=axes[1], rot=15, color='steelblue', legend=False)
axes[1].set_title('RMSE, регрессия')
axes[1].set_ylabel('RMSE')

results.set_index('Модель')['R2'].plot(kind='bar', ax=axes[2], rot=15, color='darkorange', legend=False)
axes[2].set_title('R2, регрессия')
axes[2].set_ylim(0, 1.05)
axes[2].set_ylabel('R2')

plt.tight_layout()
plt.show()


## Выводы

На классификации все три столбца высокие и почти одной высоты. Улучшенный вариант чуть выше остальных, своя реализация от бейзлайна почти не отличается.

На регрессии столбцы RMSE у всех длинные и одинаковые, а R2 у всех низкий. По графику разницы между бейзлайном, улучшенной моделью и своей реализацией нет.
